In [22]:
from langgraph.graph import  Graph
from langchain_community.embeddings import  HuggingFaceBgeEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain.prompts import ChatPromptTemplate
from langchain.llms import LLM
from langchain.chains import RetrievalQA
from langchain.runnables import RunnablePassthrough
from langchain.output_parsers import StrOutputParser


from dotenv import  load_dotenv
load_dotenv()

ImportError: cannot import name 'LLM' from 'langchain.llms' (/Users/rahulprajapati/miniconda3/envs/llm/lib/python3.10/site-packages/langchain/llms/__init__.py)

In [17]:
def func1(input_1):
    try:
        return input_1 + 'from first function '
    except Exception as e:
        print(f'error raise from func1 :: {e}')



def func2(input_2):
    try:
        return input_2 + 'from second function'
    except Exception as e:
        print(f'error raise from func2 :: {e}')
        

In [18]:

workflow = Graph()

# adding the node:
workflow.add_node('node_1', func1)
workflow.add_node('node_2', func2)

# adding the edge:
workflow.add_edge('node_1', 'node_2')

# configure entery and exist point
workflow.set_entry_point('node_1')
workflow.set_finish_point('node_2')

# compile the workflow
app = workflow.compile()




In [19]:
app.invoke('I am runing from')

'I am runing fromfrom first function from second function'

## Intergating LLm call in this Graph:

In [7]:
from langchain_community.llms import Ollama

In [9]:
llm = Ollama(model = 'llama3.2:1b')

In [55]:
def func1(input_1):
    try:
        complete_query = "ArithmeticErrorYour task is to provide the topic based on user query \
            and the topin belongs to [Japan, sport]. no used any reasoing for this query." + input_1
        response = model.invoke(complete_query)
        return response
    except Exception as e:
        print(f"Error: {e}")

In [56]:
def func2(input_2):
    try:
        upper_case = input_2.upper()
        return f' this output from the func2 {upper_case}'
    except Exception as e:
        print(f'error ::{e}')

In [57]:
# define the workflow:

workflow = Graph()

workflow.add_node('Agent', func1)
workflow.add_node('tool', func2)

workflow.add_edge('Agent', 'tool')

workflow.set_entry_point('Agent')
workflow.set_finish_point('tool')

app = workflow.compile()


In [58]:
query = 'tell me about virat kohli'
app.invoke(query)

' this output from the func2 USER QUERY:\n\nTOPIC: VIRAT KOHLI\n\nTHE TOP RESULT FOR "VIRAT KOHLI" RELATED TO JAPAN IS NOT AVAILABLE AS "JAPAN" IS NOT DIRECTLY RELATED TO VIRAT KOHLI. HOWEVER, I CAN PROVIDE INFORMATION ON VIRAT KOHLI\'S BACKGROUND AND CAREER IN RELATION TO JAPAN.\n\nVIRAT KOHLI IS AN INDIAN CRICKETER WHO HAS REPRESENTED INDIA IN INTERNATIONAL CRICKET. HE WAS BORN ON AUGUST 5, 1988, IN AMRITSAR, PUNJAB, INDIA. KOHLI PLAYED FOR THE DELHI CAPITALS IN THE INDIAN PREMIER LEAGUE (IPL) BEFORE MAKING HIS INTERNATIONAL DEBUT IN 2011.\n\nKOHLI\'S CRICKET CAREER HAS TAKEN HIM TO VARIOUS COUNTRIES, INCLUDING ENGLAND, AUSTRALIA, AND NEW ZEALAND. HE IS KNOWN FOR HIS EXCEPTIONAL BATTING SKILLS AND RECORD-BREAKING PERFORMANCES IN INTERNATIONAL CRICKET.\n\nTHERE ARE NO DIRECT CONNECTIONS BETWEEN VIRAT KOHLI AND JAPAN.'

/Users/rahulprajapati/Documents/GitHub/LangGraph/Graph_01


### RAG Pipeline Inntergation with langgraph:

In [20]:

## load documents:

loader = DirectoryLoader(
    '/Users/rahulprajapati/Documents/GitHub/LangGraph/data', glob = './*.txt',
    loader_cls = TextLoader
)

docs = loader.load()

# print(docs)
# print(len(docs))

# create chunks out of this documents:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 50,
    length_function = len
)

new_docs  = text_splitter.split_documents(docs)
# print(len(new_docs))

# indxing of the docuements:


model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
hf = HuggingFaceBgeEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)


/Users/rahulprajapati/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:

db = Chroma.from_documents(new_docs, embedding=hf)
retriever = db.as_retriever(search_kwargs = {'k':3})


In [22]:
# adding the Agent State concept which is accessable for all the node of this given graph:

AgentState = {}
AgentState['message'] = []

In [39]:
from langchain.chains import RetrievalQA

In [47]:

def func1(state):
    try:
        message = state['message']
        question = message[-1]

        complete_query = "your task is to provide the topic based on user query \
            and the topic is belongs to ['technology', 'person] of this question "+ question

        response = llm.invoke(complete_query)
        state['message'].append(response)
        return state
    except Exception as e:
        print(f'error is raised from func1 :: {e}')
        
def func2(state):
    try:
        message = state['message']
        question = message[0]  # i am picking the first question.

        prompt_temp = """
        Answer the following question based on the question

        {context}

        Question : {question}
        """
        
        prompt = ChatPromptTemplate.from_template(prompt_temp)

        # Assuming retriever is an object already set up
        retriever_chain = RetrievalQA.from_chain_type(
            llm=llm,  # The LLM to be used (ensure it's initialized earlier)
            chain_type="map_reduce",  # You can adjust this depending on your chain type
            retriever=retriever,  # Assuming 'retriever' is set up correctly earlier
            return_source_documents=True,
        )

        # Invoking the retriever chain
        result = retriever_chain.invoke({'query': question})
        print(result)
        return result

    except Exception as e:
        print(f'error is raised from func2 :: {e}')

In [48]:
## create graph

workflow = Graph()

workflow.add_node('Agent', func1)
workflow.add_node('RAG', func2)

workflow.add_edge('Agent', 'RAG')

workflow.set_entry_point('Agent')
workflow.set_finish_point('RAG')

app = workflow.compile()


In [49]:
inp = {'message': ['tell me about virat kohli']}
app.invoke(inp)

Token indices sequence length is longer than the specified maximum sequence length for this model (1747 > 1024). Running this sequence through the model will result in indexing errors


{'query': 'tell me about virat kohli', 'result': "Based on the provided portion of the document, here is a final answer:\n\nThere is no mention of Virat Kohli in the given text. The information provided only describes his cricket career and statistics.\n\nIf you're looking for information about Virat Kohli, I can try to help you find relevant text if you provide more context or details about what you're interested in learning more about.", 'source_documents': [Document(metadata={'source': '/Users/rahulprajapati/Documents/GitHub/LangGraph/data/virat_kohli.txt'}, page_content='Virat Kohli (Hindi pronunciation: [ʋɪˈɾɑːʈ ˈkoːɦli] ⓘ; born 5 November 1988) is an Indian'), Document(metadata={'source': '/Users/rahulprajapati/Documents/GitHub/LangGraph/data/virat_kohli.txt'}, page_content='bowler. Kohli holds the highest IPL run-scorer record, ranks third in T20I, third in ODI, and'), Document(metadata={'source': '/Users/rahulprajapati/Documents/GitHub/LangGraph/data/virat_kohli.txt'}, page_con

{'query': 'tell me about virat kohli',
 'result': "Based on the provided portion of the document, here is a final answer:\n\nThere is no mention of Virat Kohli in the given text. The information provided only describes his cricket career and statistics.\n\nIf you're looking for information about Virat Kohli, I can try to help you find relevant text if you provide more context or details about what you're interested in learning more about.",
 'source_documents': [Document(metadata={'source': '/Users/rahulprajapati/Documents/GitHub/LangGraph/data/virat_kohli.txt'}, page_content='Virat Kohli (Hindi pronunciation: [ʋɪˈɾɑːʈ ˈkoːɦli] ⓘ; born 5 November 1988) is an Indian'),
  Document(metadata={'source': '/Users/rahulprajapati/Documents/GitHub/LangGraph/data/virat_kohli.txt'}, page_content='bowler. Kohli holds the highest IPL run-scorer record, ranks third in T20I, third in ODI, and'),
  Document(metadata={'source': '/Users/rahulprajapati/Documents/GitHub/LangGraph/data/virat_kohli.txt'}, pa

### Complex Degine work flow throw the Lnaggraph:
#### adding the condtional router.

In [15]:
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_core.messages import  BaseMessage


In [16]:
class AgentState(TypedDict):
    message = Annotated[Sequence[BaseMessage], operator.add]

In [1]:
from pydantic import BaseModel, Field


In [13]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

from langchain.prompts import PromptTemplate


class TopicSelectionParser(BaseModel):
    Topic: str = Field(..., description='Select Topic')
  

In [14]:

parser = PydanticOutputParser(pydantic_object=TopicSelectionParser)

In [20]:
def func1(state):

    try:
        print('Calling LLM Agent  --->>')
        message = state['message']
        quesiton = message[-1]

        template = """ your task is to provide topic related to the user query \
            only output the topic among = ['japan', 'sport', 'not defined'] do not use the reasoing for following qery:
            {question}{format_instructions} 
        
        """

        prompt = PromptTemplate(
            template = template,
            input_variable = ['question'],
            partial_variables ={
                'format_instructions': parser.get_format_instructions()
            } 
        )

        chain = prompt | llm | parser

        result = chain.invoke('question', question, 'format_instructions',  parser.get_format_instructions())
        return {'message': [result.topic]}
    except Exception as e:
        print(f"error raise from function func1 :: {e}", )


In [23]:
def func2(state):
    try:
        print('Calling RAG -->>')
        message = state['message']
        question = message[0]

        prompt_templare = """ 
        Answer the following question based on given context:

        {context}

        Question :{question}
        """

        prompt = ChatPromptTemplate.from_template(template=prompt_templare)

        # Assuming retriever is an object already set up
        retriever_chain = RetrievalQA.from_chain_type(
            llm=llm,  # The LLM to be used (ensure it's initialized earlier)
            chain_type="map_reduce",  # You can adjust this depending on your chain type
            retriever=retriever,  # Assuming 'retriever' is set up correctly earlier
            return_source_documents=True,
        )

        # Invoking the retriever chain
        result = retriever_chain.invoke({'query': question})
        return {'message': [result]}


    except Exception as e:
        print(f"error raise from in func2: {e}")

In [ ]:

def func3(state):
    try:
        print('calling  bydefualt llm  --->>')
        message = state['message']
        question = message[0]
        temp = "Answer the following quesiton with your knowleged of real world, following user question:" + question
        response = llm.invoke(temp)
        return {'message': [response]}
    except Exception as e:
        print(f'error is raise from func3 :: {e}')
